In [1]:
import os
import json
import numpy as np
import joblib
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout, BatchNormalization, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
EPOCHS = 30
BATCH_SIZE = 256
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
X = np.load("/content/drive/MyDrive/NIDS/X_dcnn (1).npy")
y = np.load("/content/drive/MyDrive/NIDS/y_labels (1).npy")

with open("/content/drive/MyDrive/NIDS/class_mapping.json") as f:
    class_mapping = json.load(f)

label_names = [class_mapping[str(i)] for i in range(len(class_mapping))]
n_features = X.shape[1]
n_classes = len(label_names)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {label_names}")

Loading preprocessed data...
X shape: (239556, 78, 1)
y shape: (239556,)
Classes: ['BENIGN', 'Botnet', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'PortScan', 'SSH-Patator', 'Web Attacks']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=SEED
)

print("\nSplit sizes:")
print(f"Train: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")


Split sizes:
Train: 172479
Val:   19165
Test:  47912


In [5]:
def build_model(n_features, n_classes):
    model = Sequential([
        tf.keras.layers.InputLayer(input_shape=(n_features, 1)),

        Conv1D(128, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),

        Conv1D(256, 3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(2),

        Conv1D(256, 3, activation="relu", padding="same"),
        BatchNormalization(),

        Flatten(),

        Dense(512, activation="relu"),
        Dropout(0.3),

        Dense(256, activation="relu"),
        Dropout(0.2),

        Dense(n_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model(n_features, n_classes)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 78, 128)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 39, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 39, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 39, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 19, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 19, 256)        │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 19, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4864)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     2,490,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 11)             │         2,827 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,923,531 (11.15 MB)

 Trainable params: 2,922,251 (11.15 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [6]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - accuracy: 0.9627 - loss: 0.1275 - val_accuracy: 0.7837 - val_loss: 0.7782 - learning_rate: 1.0000e-04
Epoch 2/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9855 - loss: 0.0497 - val_accuracy: 0.9875 - val_loss: 0.0409 - learning_rate: 1.0000e-04
Epoch 3/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9875 - loss: 0.0420 - val_accuracy: 0.9884 - val_loss: 0.0372 - learning_rate: 1.0000e-04
Epoch 4/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9889 - loss: 0.0376 - val_accuracy: 0.9901 - val_loss: 0.0329 - learning_rate: 1.0000e-04
Epoch 5/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9904 - loss: 0.0325 - val_accuracy: 0.9920 - val_loss: 0.0275 - learning_rate: 1.0000e-04
Epoch 6/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9915 - loss: 0.0301 - val_accuracy: 0.9902 - val_loss: 0.0314 - learning_rate: 1.0000e-04
Epoch 7/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - ac

In [7]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}")


Test Accuracy: 0.9948


In [10]:
SAVE_DIR = "/content/drive/MyDrive/NIDS"
os.makedirs(SAVE_DIR, exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(SAVE_DIR, "nids_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"Saved TFLite model: {tflite_path}")

labels_json = {
    "labels": label_names,
    "index_to_label": {str(i): name for i, name in enumerate(label_names)},
    "n_classes": n_classes,
    "n_features": n_features
}

with open(os.path.join(SAVE_DIR, "class_labels.json"), "w") as f:
    json.dump(labels_json, f, indent=2)

print("\nTraining complete. TFLite model saved in Google Drive → NIDS folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved artifact at '/tmp/tmpl4tituer'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 78, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  134638280772688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280773456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280775376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280775760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280772304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280774992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280774800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280776144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134638280776528: TensorSpec(sh